## Step 1 - Load Features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

features_dir = Path().resolve().parent / 'data' / 'features'
features = pd.read_parquet(features_dir / 'features.parquet')

# Cast condition_cluster from category to str immediately after load.
# Parquet preserves category dtype from Stage 2, and pandas Categorical scalars
# do not reliably match plain string dict keys in .get() calls — causing silent
# fallback to BASE_RATES['Healthy'] for some clusters (confirmed: MSK=0.0% bug).
features['condition_cluster'] = features['condition_cluster'].astype(str)

print(f"Features loaded: {features.shape}")
print(f"\nCondition cluster distribution (confirm 'Healthy' not 'None'):")
print(features['condition_cluster'].value_counts())
print(f"\nZero allied health flag rate: {features['zero_allied_health_flag'].mean():.1%}")
print(f"High GP + low allied rate:    {features['high_gp_low_allied'].mean():.1%}")

## Step 2 - Define Base Rates and Amplifiers

Base rates represent the unconditional probability of a member in that condition cluster having an acute event in the forward window. Amplifiers are multiplicative — they stack on top of the base rate when risk factors are present.

In [ ]:
# Base probability of acute event by condition cluster
# 'Healthy' = no condition flags — background rate only
# Higher clusters reflect clinical literature on unmanaged chronic condition escalation
BASE_RATES = {
    'Healthy':   0.04,   # background rate — no condition flags
    'MSK':       0.18,   # unmanaged back/joint pain escalating to ED
    'Metabolic': 0.15,   # unmanaged DM/obesity complications
    'MH':        0.12,   # MH crisis presenting to ED
    'Mixed':     0.28,   # 2+ condition clusters — compounding risk
}

# Multiplicative risk amplifiers
# Each amplifier applies independently; they stack multiplicatively
# Cap at 0.95 to avoid deterministic labels
AMPLIFIERS = {
    'age_over_55':       1.30,  # older members have higher escalation risk
    'zero_allied_6m':    1.40,  # core nudge signal — no allied health use in 6m
    'high_gp_low_allied':1.20,  # seeking care via GP but not through allied health
    'bronze_plan':       1.15,  # lower benefit entitlement reduces access
    'comorbidity_2plus': 1.25,  # 2+ condition clusters compounds risk
}

## Step 3 - Simulate the labels

Each member gets a probability calculated from their base rate multiplied by whichever amplifiers apply to them. A single Bernoulli draw then assigns the binary label. `np.random.seed(42)` ensures reproducibility 

In [ ]:
np.random.seed(42)

def simulate_acute_risk(row):
    # Look up base rate for this member's condition cluster
    # Falls back to 'Healthy' rate if an unexpected cluster value appears
    # str() cast is defensive belt-and-suspenders — Step 1 already casts the
    # column, but this ensures correctness if the function is ever called with
    # a raw parquet row before the Step 1 cast has been applied.
    base = BASE_RATES.get(str(row['condition_cluster']), BASE_RATES['Healthy'])

    # Extra boost for triple comorbidity (all three flags set)
    if row['comorbidity_count'] == 3:
        base = min(base * 1.15, 0.95)

    # Stack amplifiers multiplicatively
    multiplier = 1.0
    if row['age'] > 55:
        multiplier *= AMPLIFIERS['age_over_55']
    if row['zero_allied_health_flag'] == 1:
        multiplier *= AMPLIFIERS['zero_allied_6m']
    if row['high_gp_low_allied'] == 1:
        multiplier *= AMPLIFIERS['high_gp_low_allied']
    if row['plan_type'] == 'Bronze':
        multiplier *= AMPLIFIERS['bronze_plan']
    if row['comorbidity_count'] >= 2:
        multiplier *= AMPLIFIERS['comorbidity_2plus']

    # Final probability — capped at 0.95 to avoid deterministic labels
    prob = min(base * multiplier, 0.95)
    return int(np.random.binomial(1, prob))

labels = features[['member_id']].copy()
labels['high_acute_risk'] = features.apply(simulate_acute_risk, axis=1)

pos_count = labels['high_acute_risk'].sum()
pos_rate  = labels['high_acute_risk'].mean()

print("Label distribution:")
print(labels['high_acute_risk'].value_counts())
print(f"\nPositive class rate: {pos_rate:.1%}  ({pos_count:,} members flagged)")

# Compute scale_pos_weight now — needed in Stage 4
neg_count = (labels['high_acute_risk'] == 0).sum()
scale_pos_weight = neg_count / pos_count
print(f"\nscale_pos_weight for LightGBM (Stage 4): {scale_pos_weight:.2f}")
print(f"  → {neg_count:,} negatives / {pos_count:,} positives")

## Step 4 - Sanity Checks

Verify label is directionally correct

In [ ]:
check = features.merge(labels, on='member_id')

print("=== 1. Positive rate by condition cluster ===")
cluster_rates = check.groupby('condition_cluster', observed=True)['high_acute_risk'].mean().sort_values(ascending=False)
print(cluster_rates)

print("\n=== 2. Positive rate by age band ===")
age_rates = check.groupby('age_band', observed=True)['high_acute_risk'].mean().sort_values(ascending=False)
print(age_rates)

print("\n=== 3. Positive rate: zero allied vs any allied health ===")
allied_rates = check.groupby('zero_allied_health_flag')['high_acute_risk'].mean()
print(allied_rates)
print(f"  Uplift from zero allied: {allied_rates[1] / allied_rates[0]:.2f}x")

print("\n=== 4. Positive rate by plan type ===")
plan_rates = check.groupby('plan_type')['high_acute_risk'].mean().sort_values(ascending=False)
print(plan_rates)

print("\n=== 5. Positive rate: high_gp_low_allied flag ===")
gp_allied_rates = check.groupby('high_gp_low_allied')['high_acute_risk'].mean()
print(gp_allied_rates)